<a href="https://colab.research.google.com/github/dammy-idowu/Healthcare-AI/blob/main/Breast_cancer_prediction_from_Mammogram_images_using_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Deep Learning-Based Predictive Analytics for Automated Mammographic Breast Cancer Categorization**

### **Introduction**
Breast cancer remains one of the leading causes of oncological mortality among women globally and across developing nations like Nigeria, where access to specialized clinical radiographers is severely constrained. Traditional diagnostic workflows rely heavily on manual interpretation of mammographic imaging, a process highly susceptible to human fatigue, inter-observer variability and delayed diagnostic turnaround times.

To bridge this healthcare delivery gap, digital health infrastructure and artificial intelligence are being positioned as critical screening mechanisms. Convolutional Neural Networks (CNNs) offer a scalable, highly precise solution for automated medical image analysis. By processing pixel-level structural patterns, microcalcifications and architectural distortions invisible to the naked human eye, deep learning architectures can rapidly analyze digital mammograms. Implementing these automated diagnostic systems acts as a vital triage tool, drastically reducing diagnostic latency, easing the burden on sparse oncology workloads and facilitate early intervention which is crucial for improved patient survival rate.


## **Project Objective**
The primary objective of this predictive analytics project is to develop, train and validate a deep learning pipeline capable of classifying mammographic breast tissue anomalies into three distinct clinical profiles: Normal, Benign and Malignant.

Specifically, this technical workflow aims to:

- Implement a Stratified Machine Learning Pipeline: Architect a reproducible data pipeline that overcomes data selection bias by executing stratified splits (70% Train, 15% Validation, 15% Test) to ensure balanced representation across all diagnostic sub-categories.

- Leverage Advanced Transfer Learning: Optimize a pre-trained ResNet-18 CNN architecture to act as a high-performance feature extractor, adapting its final fully connected layer to compute multi-class target probabilities on a localized mammogram dataset (Dataset_BUSI_with_GT).

- Enforce Rigorous Model Validation: Mitigate over-confident weight metrics and artificial accuracy traps by integrating Label Smoothing Cross-Entropy Loss and embedding a real-time Class-Distribution Audit Tracker to verify multi-category training ingestion.

- Evaluate Diagnostic Clinical Metrics: Benchmark the finalized diagnostic engine against unseen validation and test data partitions, measuring success through micro-averaged precision, recall, F1-scores and a structured confusion matrix.

### **`Environmental Configuration & Dependency Initialization`**

In [ ]:
# Install library
!pip install optuna      # for Hyperparameter tuning

In [ ]:
# Import required libraries for ML operation
import os
import optuna
import zipfile
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset, Subset
import copy

In [ ]:
# Set device to Google free cloud T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running with hardware acceleration on: {device}")

Running with hardware acceleration on: cuda


### **`Mammographic Data collection and Inspection`**

In [ ]:
# Mount Google Drive to Colab Environment
from google.colab import drive
drive.mount('/content/drive')

# Map mammogram data zipped file directory from Google Drive
zip_path = '/content/drive/My Drive/Healthcare AI/mammogram_dataset.zip'
extract_path = '/content/mammogram_data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Extract mammographic data content from the Zip file
print("Extracting mammogram data (images)  from Zipfile... please wait.")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Unzipping complete! Mammogram images are ready at:", extract_path)

# Navigate path to find the target subfolders from extracted data
true_data_root = os.path.join(extract_path, 'Dataset_BUSI_with_GT')

# Double-check if the folder actually exists at this sub-path
if not os.path.exists(true_data_root):
    # Fallback if the folder name capitalization is slightly different
    subfolders = [f for f in os.listdir(extract_path) if os.path.isdir(os.path.join(extract_path, f))]
    if len(subfolders) == 1:
        true_data_root = os.path.join(extract_path, subfolders[0])
    else:
        true_data_root = extract_path

print(f"\n[CORRECTED] Target path redirected to: {true_data_root}")
print("Actual subfolders found inside target path:", os.listdir(true_data_root))

# Standard Basic Transform for initial inspection of mammogram data
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load dataset pointing to the true internal root
try:
    full_dataset = datasets.ImageFolder(root=true_data_root, transform=basic_transform)
    print("\n================ PIPELINE RE-INITIALIZED ================")
    print(f"Total mammogram images detected: {len(full_dataset)}")
    print(f"Corrected Class Folders: {full_dataset.classes}")
    print(f"Corrected Class Mapping: {full_dataset.class_to_idx}")
except Exception as e:
    print(f"\n[ERROR] Loading failed. Please verify the names of the folders inside {true_data_root}")
    print(f"Details: {e}")


Extracting mammogram data (images)  from Zipfile... please wait.
Unzipping complete! Mammogram images are ready at: /content/mammogram_data

[CORRECTED] Target path redirected to: /content/mammogram_data/Dataset_BUSI_with_GT
Actual subfolders found inside target path: ['normal', 'benign', 'malignant']

================ PIPELINE RE-INITIALIZED ================
Total mammogram images detected: 1578
Corrected Class Folders: ['benign', 'malignant', 'normal']
Corrected Class Mapping: {'benign': 0, 'malignant': 1, 'normal': 2}


### **`Data Preprocessing & Stratified Splitting`**

In [ ]:
# Mammogram data transformation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val_test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Explicitly target internal directory to avoid single class caching
true_data_root = '/content/mammogram_data/Dataset_BUSI_with_GT'
full_dataset = datasets.ImageFolder(root=true_data_root)

targets = full_dataset.targets
indices = np.arange(len(full_dataset))

# # Set random seed for reproducibility across splits
# torch.manual_seed(42)
# np.random.seed(42)

# Stratify splits: 70% Train, 15% Validation, 15% Test
train_idx, temp_idx, _, temp_targets = train_test_split(
    indices, targets, test_size=0.30, stratify=targets, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_targets, random_state=42
)

# Apply subset wrapper to preserve structural layout transforms
class TransformedSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.dataset[self.indices[index]]
        if self.transform: x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.indices)

train_dataset = TransformedSubset(full_dataset, train_idx, transform=data_transforms['train'])
val_dataset = TransformedSubset(full_dataset, val_idx, transform=data_transforms['val_test'])
test_dataset = TransformedSubset(full_dataset, test_idx, transform=data_transforms['val_test'])

# Create DataLoaders for streaming balanced mini-batches
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2),
    'val': DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2),
    'test': DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
}

print(f"\n[SUCCESS] Balanced Dataset Splits Created:")
print(f"-> Train Set: {len(train_dataset)} images")
print(f"-> Validation Set: {len(val_dataset)} images")
print(f"-> Test Set: {len(test_dataset)} images")

# index-to-class text mapping dictionary for mammogram diagnostic outcome
idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}

print("Mapping successfully defined!")
print(f"Mammographic breast tissue clinical profile mapping: {idx_to_class}")



[SUCCESS] Balanced Dataset Splits Created:
-> Train Set: 1104 images
-> Validation Set: 237 images
-> Test Set: 237 images
Mapping successfully defined!
Mammographic breast tissue clinical profile mapping: {0: 'benign', 1: 'malignant', 2: 'normal'}


### **`Model Architecture Setup & Training`**

In [ ]:
# Define the Objective Search Function for Optuna
def objective(trial):
    # Optuna will pick different values for these parameters in each trial

    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.2)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'RMSprop'])

    # Initialize ResNet18 model architecture
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 3)
    model = model.to(device)

    # Inject chosen parameters into loss criterion and optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=lr)

    # Run fast 2-epoch evaluation sweep per trial
    for epoch in range(2):
        model.train()
        for inputs, labels in dataloaders['train']:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation Phase
        model.eval()
        running_corrects = 0
        total_val_samples = 0

        with torch.no_grad():
            for inputs, labels in dataloaders['val']:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                running_corrects += torch.sum(preds == labels.data)
                total_val_samples += labels.size(0)

        val_accuracy = running_corrects.double() / total_val_samples

    # Return validation accuracy back to Optuna engine to assess choice strength
    return val_accuracy

# Execute the Search Study to Maximise Accuracy
print("Executing Automated Optimization Sweeps...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

print("\n================ HYPERPARAMETER TUNING COMPLETE ================")
print(f"Highest Validation Accuracy Discovered: {study.best_value:.4f}")
print(f"Optimal Parameters: {study.best_params}")

# 3. Train Final Model Using the Highest Performance Parameters
print("\nTraining final production model using optimal configurations...")
best_lr = study.best_params['lr']
best_smoothing = study.best_params['label_smoothing']
best_opt_name = study.best_params['optimizer']

# Re-build model structure for full training
trained_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
trained_model.fc = nn.Linear(trained_model.fc.in_features, 3)
trained_model = trained_model.to(device)

final_criterion = nn.CrossEntropyLoss(label_smoothing=best_smoothing)
if best_opt_name == 'Adam':
    final_optimizer = optim.Adam(trained_model.parameters(), lr=best_lr)
else:
    final_optimizer = optim.RMSprop(trained_model.parameters(), lr=best_lr)

# Run full production training loops with optimal parameters
for epoch in range(5):
    trained_model.train()
    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)
        final_optimizer.zero_grad()
        loss = final_criterion(trained_model(inputs), labels)
        loss.backward()
        final_optimizer.step()

print("Final model optimized and ready for testing evaluation!")

[I 2026-05-24 00:49:17,384] A new study created in memory with name: no-name-e1e73bee-0802-4d05-8dbf-1bbbb5064379


Executing Automated Optimization Sweeps...


[I 2026-05-24 00:49:37,839] Trial 0 finished with value: 0.8481012658227848 and parameters: {'lr': 5.187160735301818e-05, 'label_smoothing': 0.1397959927572536, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.8481012658227848.
[I 2026-05-24 00:49:58,701] Trial 1 finished with value: 0.8902953586497889 and parameters: {'lr': 3.8630381442106736e-05, 'label_smoothing': 0.14357603025132046, 'optimizer': 'Adam'}. Best is trial 1 with value: 0.8902953586497889.
[I 2026-05-24 00:50:19,493] Trial 2 finished with value: 0.8523206751054851 and parameters: {'lr': 1.780578880578738e-05, 'label_smoothing': 0.18959446359640306, 'optimizer': 'Adam'}. Best is trial 1 with value: 0.8902953586497889.
[I 2026-05-24 00:50:39,330] Trial 3 finished with value: 0.729957805907173 and parameters: {'lr': 0.0007230731291309938, 'label_smoothing': 0.12039004112927526, 'optimizer': 'Adam'}. Best is trial 1 with value: 0.8902953586497889.
[I 2026-05-24 00:51:01,436] Trial 4 finished with value: 0.83544303797


================ HYPERPARAMETER TUNING COMPLETE ================
Highest Validation Accuracy Discovered: 0.8903
Optimal Parameters: {'lr': 3.8630381442106736e-05, 'label_smoothing': 0.14357603025132046, 'optimizer': 'Adam'}

Training final production model using optimal configurations...
Final model optimized and ready for testing evaluation!


### **`Model Performance Evaluation & Multi-Class Metrics Assessment of Mammographic data`**

In [ ]:
# Model performance evaluation on test set
trained_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        outputs = trained_model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Map predictions back to original diagnostic categories
target_names = list(full_dataset.class_to_idx.keys())

print("==== EVALUATION REPORT FOR MACHINE LEARNING PREDICTIVE MODEL ====")
print(classification_report(all_labels, all_preds, target_names=target_names))

print("==== CONFUSION MATRIX ====")
print(confusion_matrix(all_labels, all_preds))


==== EVALUATION REPORT FOR MACHINE LEARNING PREDICTIVE MODEL ====
              precision    recall  f1-score   support

      benign       0.88      0.96      0.92       134
   malignant       1.00      0.84      0.91        63
      normal       0.84      0.78      0.81        40

    accuracy                           0.90       237
   macro avg       0.91      0.86      0.88       237
weighted avg       0.90      0.90      0.90       237

==== CONFUSION MATRIX ====
[[129   0   5]
 [  9  53   1]
 [  9   0  31]]


### **`Inference`**

##### `Hyperparameter tuning`
- After hyperparameter tuning, Highest Validation Accuracy Discovered was: 0.8903 \
Optimal Parameters: {'lr': 3.8630381442106736e-05, 'label_smoothing': 0.14357603025132046, 'optimizer': 'Adam'}; were ultimately used for training the model for predictive analytics.


##### `Classification report and Performance metric evaluation`

Actual Benign (Row 1): 134 cases
- 129 were correctly identified as benign.
- 0 were wrongly predicted as malignant.
- 5 were wrongly predicted as normal.

Actual Malignant (Row 2): 63 cases
- 53 were correctly caught.
- 9 were missed and misclassified as benign.
- 1 was missed and misclassified as normal.

Actual Normal (Row 3): 40 cases
- 31 were correctly identified as healthy/normal tissue.
- 9 were wrongly predicted as benign.
- 0 were wrongly predicted as malignant.

- The numbers down the confusion matrix diagonal (\(129\), \(53\), \(31\)) represent correct predictions. Out of 237 total test cases, 213 were correct.

- In medical predictive analytics, low recall renders risk models blind to critical events. When a model suffers from low recall, it fails to identify true positive cases, generating dangerous false negatives that cause clinicians to miss life-threatening condition.

- In this analysis, Class 1 (malignant class) has a Recall score of 0.84. This implies that the model missed 16% of breast tissue malignant cases (10 out of 63) as observed from the confusion matrix. In a clinical setting this poses a great risk. **Hence, this facilitates the need to re-train the model and optimize specifically for Malignant recall**.




In [ ]:
# Import library to optimize recall via weighted cross-entropy
from sklearn.metrics import recall_score

# Calculate base data inverse frequencies to handle natural data distributions
all_train_targets = [full_dataset.targets[i] for i in train_idx]
class_counts = np.bincount(all_train_targets)
total_samples = len(all_train_targets)

# Compute baseline inverse frequency weights
base_weights = total_samples / (len(class_counts) * class_counts)

def recall_focused_objective(trial):
    # Search space for standard hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.15)

    # Optuna will test penalizing malignant errors from 1x (normal) up to 5x heavier
    malignant_penalty_multiplier = trial.suggest_float('malignant_multiplier', 1.0, 5.0)

    # Construct dynamic, optimized class weights tensor for Malignant Class
    current_weights = np.copy(base_weights)
    current_weights[1] = current_weights[1] * malignant_penalty_multiplier
    tensor_weights = torch.FloatTensor(current_weights).to(device)

    # Initialize model architecture
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 3)
    model = model.to(device)

    # Pass the custom weights tensor into the cross-entropy configuration
    criterion = nn.CrossEntropyLoss(weight=tensor_weights, label_smoothing=label_smoothing)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Conduct 2-epoch trial pass
    for epoch in range(2):
        model.train()
        for inputs, labels in dataloaders['train']:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()

        # Validation evaluation focused on Class 1 Recall
        model.eval()
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for inputs, labels in dataloaders['val']:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        # Target Metric Optimization: Compute individual sensitivity for Malignant profiles (Class 1)
        malignant_recall = recall_score(val_labels, val_preds, labels=[1], average='macro', zero_division=0)

    # Return malignant recall to the optimization engine to force high sensitivity selection
    return malignant_recall

# Run the specialized optimization search maximizing malignant sensitivity
print("Executing Recall-Focused Optimization Sweeps...")
recall_study = optuna.create_study(direction="maximize")
recall_study.optimize(recall_focused_objective, n_trials=5)

print("\n================ SENSITIVITY OPTIMIZATION COMPLETE ================")
print(f"Highest Malignant Recall Discovered: {recall_study.best_value:.4f}")
print(f"Optimal Parameters: {recall_study.best_params}")


[I 2026-05-24 00:51:45,810] A new study created in memory with name: no-name-04e8eb2b-dc71-4570-917c-3e65d9de3db9


Executing Recall-Focused Optimization Sweeps...


[I 2026-05-24 00:52:06,870] Trial 0 finished with value: 0.8253968253968254 and parameters: {'lr': 9.58242777016442e-05, 'label_smoothing': 0.021356924162409335, 'malignant_multiplier': 1.663899618037088}. Best is trial 0 with value: 0.8253968253968254.
[I 2026-05-24 00:52:28,677] Trial 1 finished with value: 0.8571428571428571 and parameters: {'lr': 0.0008400894276160286, 'label_smoothing': 0.07621903474663567, 'malignant_multiplier': 4.721263385952284}. Best is trial 1 with value: 0.8571428571428571.
[I 2026-05-24 00:52:49,935] Trial 2 finished with value: 0.8888888888888888 and parameters: {'lr': 3.322265152709386e-05, 'label_smoothing': 0.06452975057637009, 'malignant_multiplier': 3.39351314695236}. Best is trial 2 with value: 0.8888888888888888.
[I 2026-05-24 00:53:10,239] Trial 3 finished with value: 0.7301587301587301 and parameters: {'lr': 1.2418712859186001e-05, 'label_smoothing': 0.14578790079729192, 'malignant_multiplier': 1.3459455911425438}. Best is trial 2 with value: 0.8


================ SENSITIVITY OPTIMIZATION COMPLETE ================
Highest Malignant Recall Discovered: 0.8889
Optimal Parameters: {'lr': 3.322265152709386e-05, 'label_smoothing': 0.06452975057637009, 'malignant_multiplier': 3.39351314695236}


In [ ]:
# Re-train final production model using the best sensitivity weights
print("\nTraining final recall-optimized model using optimal weights...")
best_lr = recall_study.best_params['lr']
best_smoothing = recall_study.best_params['label_smoothing']
best_multiplier = recall_study.best_params['malignant_multiplier']

final_weights = np.copy(base_weights)
final_weights[1] = final_weights[1] * best_multiplier
final_tensor_weights = torch.FloatTensor(final_weights).to(device)

tuned_recall_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
tuned_recall_model.fc = nn.Linear(tuned_recall_model.fc.in_features, 3)
tuned_recall_model = tuned_recall_model.to(device)

final_criterion = nn.CrossEntropyLoss(weight=final_tensor_weights, label_smoothing=best_smoothing)
final_optimizer = optim.Adam(tuned_recall_model.parameters(), lr=best_lr)

# Execute final training loops
for epoch in range(4):
    tuned_recall_model.train()
    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)
        final_optimizer.zero_grad()
        loss = final_criterion(tuned_recall_model(inputs), labels)
        loss.backward()
        final_optimizer.step()

print("Recall-optimized model fully trained and Ready for evaluation on test set")



Training final recall-optimized model using optimal weights...
Recall-optimized model fully trained and Ready for evaluation on test set


In [ ]:
# Re-trained model performance evaluation

# Switch the recall-optimized model to evaluation mode
tuned_recall_model.eval()

all_preds = []
all_labels = []

print("==== RUNNING EVALUATION ON SENSITIVITY-TUNED MODEL ====")

# Extract predictions using the correct recall model
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        outputs = tuned_recall_model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

# Map predictions back to original diagnostic categories
known_labels = sorted(list(full_dataset.class_to_idx.values()))
target_names = list(full_dataset.class_to_idx.keys())

print("==== EVALUATION REPORT FOR MACHINE LEARNING PREDICTIVE MODEL ====\n")
print(classification_report(all_labels, all_preds, labels=known_labels, target_names=target_names, zero_division=0))

print("==== RECALL-OPTIMIZED 3x3 CONFUSION MATRIX ====\n")
cm = confusion_matrix(all_labels, all_preds, labels=known_labels)
print(cm)


==== RUNNING EVALUATION ON SENSITIVITY-TUNED MODEL ====
==== EVALUATION REPORT FOR MACHINE LEARNING PREDICTIVE MODEL ====

              precision    recall  f1-score   support

      benign       0.98      0.67      0.80       134
   malignant       0.73      0.94      0.82        63
      normal       0.59      0.95      0.73        40

    accuracy                           0.79       237
   macro avg       0.77      0.85      0.78       237
weighted avg       0.85      0.79      0.79       237

==== RECALL-OPTIMIZED 3x3 CONFUSION MATRIX ====

[[90 21 23]
 [ 1 59  3]
 [ 1  1 38]]


#### **`Hyperparameter tuning inference`**

For the purpose of this analysis, the hyperparameter tuning succeeded in improving the model sensitivity significantly, to catching malignant cases: Malignant recall went up from 0.84 to 0.94 (accurately predicting 59 malignant cases compared to 53 cases before the hyperparameter tuning, 6 more cancer cases than before).

### **`Recommendation`**

- Integrate predictive analytics as a Decision-Support tool: The CNN model predictions must be utilized strictly to assist clinicians, rather than serve as a standalone diagnosis. Given the initial 24% false-negative rate (0.84 recall) for malignant cases, low or ambiguous AI predictions must trigger mandatory secondary validation. As observed from this analysis, which produced an improved recall score of 0.94 for better prediction of malignant cases

- Implement a Secondary Testing Protocol: For any potential breast cancer cases flagged by the model, a comprehensive clinical pathway should be initiated. Further diagnostic tests, including BRCA gene testing and contrast-enhanced MRI scans, must be conducted to confirm the definitive clinical diagnosis.

- Promote routine Mammogram Screening for women: Public health initiatives should heavily advocate for routine mammogram scans for all women within eligible age cohorts. Regular screening remains the most effective method to ensure early detection, foster health awareness, and catch sub-clinical changes before they advance.
